# Stage 5 — tight-SCS quantum screening of the cycle-polytope inequalities

This notebook implements the next step of the project:

1. regenerate the four **initial LP separators** for NS classes 27, 29, 43, and 44;
2. read every inequality saved by `Cycle_Perturbation_Search.ipynb`;
3. deduplicate **exact coefficient vectors** (not exposed faces);
4. verify exactly that every retained inequality has cycle-polytope bound `1`;
5. construct its Bell operator automatically in the canonical order
   \(p(abc\mid xyz)\);
6. compute `QuantumNPA.jl` lower bounds at hierarchy levels 2, 3, and 4;
7. rank the inequalities by the level-4 normalized value.

The inequalities have the form

\[
    \sum_{a,b,c,x,y,z} c_{abcxyz}\,p(abc\mid xyz) \ge 1.
\]

Therefore a level-4 value below `1` is the interesting direction.  It is a
**screening signal**, not by itself a constructive proof of quantum violation:
NPA minimizes over an outer relaxation of the quantum set.  A later see-saw
calculation must supply an explicit quantum realization below `1`.

The NPA construction follows Carlos's code, but the matrices are no longer
copied by hand.  The 64 exact coefficients are mapped directly to the three
parties' projector products.  The API used here is the one documented by the
[official QuantumNPA repository](https://github.com/ewoodhead/QuantumNPA.jl).

This copy preserves the complete candidate-generation, loading, exact-audit,
and ranking pipeline of the original notebook.  Its only substantive numerical
change is that every NPA problem is solved with tighter SCS tolerances.  A new
output directory is used so that the previous checkpoint cannot cause these
new solves to be skipped and the original results remain untouched.


## How to use this notebook

- Put this notebook in the same directory in which you ran
  `Cycle_Perturbation_Search.ipynb`.
- The directory `cycle_perturbation_results` must be beside the notebook and
  must contain the four files `class_27_candidates.tsv`,
  `class_29_candidates.tsv`, `class_43_candidates.tsv`, and
  `class_44_candidates.tsv`.
- In Jupyter choose **Kernel → Restart Kernel and Run All Cells**.
- The first code cell installs only packages that are missing from the active
  Julia environment.
- The NPA scan checkpoints after every completed level.  It can therefore be
  restarted without repeating completed SDPs.

The final files are written to `cycle_npa_results_tight_scs`.  The most important one is
`quantum_npa_results.tsv`; a smaller table containing only level-4 promising
candidates is written as `quantum_npa_promising.tsv`.


In [3]:
# Package bootstrap.  Existing packages are left untouched; only missing
# packages are added to the active Julia environment.
import Pkg

required_packages = [
    "JuMP",
    "MathOptInterface",
    "GLPK",
    "QuantumNPA",
    "SCS",
]

missing_packages = [
    package for package in required_packages
    if Base.find_package(package) === nothing
]

if !isempty(missing_packages)
    println("Installing missing Julia packages: ", join(missing_packages, ", "))
    Pkg.add(missing_packages)
else
    println("All required Julia packages are already available.")
end


All required Julia packages are already available.


In [4]:
using Dates
using GLPK
using JuMP
using LinearAlgebra
using Printf
using QuantumNPA
using SCS
using Serialization
using SparseArrays

import MathOptInterface as MOI

# -------------------------- USER CONFIGURATION --------------------------

# This is the output directory created by Cycle_Perturbation_Search.ipynb.
PERTURBATION_RESULTS_DIRECTORY = joinpath(pwd(), "cycle_perturbation_results")

# Results and restart checkpoint produced by this tighter-SCS rerun.  This is
# intentionally different from the original directory: otherwise the existing
# checkpoint would make the notebook skip candidate/level pairs already solved
# with the earlier SCS settings.  Keeping both directories also permits a direct
# comparison between the original and tighter numerical results.
NPA_RESULTS_DIRECTORY = joinpath(pwd(), "cycle_npa_results_tight_scs")

# Carlos evaluated levels 2, 3, and 4.  Level 4 is used for the final ranking.
NPA_LEVELS = [2, 3, 4]

# Numerical settings passed to SCS for every NPA semidefinite program.  These
# tolerances are stricter than SCS's defaults.  `max_iters` is only a ceiling:
# SCS may stop earlier when its convergence tests are satisfied.
SCS_EPS_ABS = 1e-8
SCS_EPS_REL = 1e-8
SCS_MAX_ITERS = 1_000_000

TIGHT_SCS_SOLVER = optimizer_with_attributes(
    SCS.Optimizer,
    "eps_abs" => SCS_EPS_ABS,
    "eps_rel" => SCS_EPS_REL,
    "max_iters" => SCS_MAX_ITERS,
    "verbose" => 0,
)

# If true, a rerun skips candidate/level pairs already in the checkpoint.
RESUME_NPA_SCAN = true

# Unexpected errors are checkpointed and then stop the run.  This prevents a
# partially failed research run from looking superficially complete.
STOP_ON_NPA_ERROR = true

# Values closer to 1 than this are labelled "no resolved level-4 gap".
NPA_SCREENING_TOLERANCE = 5e-4

# Small inversions between successive NPA levels can occur because SCS is a
# numerical solver.  This tolerance is used only for a diagnostic warning.
NPA_MONOTONICITY_TOLERANCE = 5e-4

# ------------------------------------------------------------------------

CLASSES = (27, 29, 43, 44)
TRIPLES = [(u, v, w) for u in 0:1 for v in 0:1 for w in 0:1]

println("Perturbation input directory: ", abspath(PERTURBATION_RESULTS_DIRECTORY))
println("NPA output directory:          ", abspath(NPA_RESULTS_DIRECTORY))


Perturbation input directory: C:\Users\josen\OneDrive\Desktop\Work\Post-doc_Recife\alternative_projects\Rabello's_project\CV_dropbox_content\Projeto_Ze_Gabrielzinho_Rabelo_Planeta\Jupiter\cycle_perturbation_results
NPA output directory:          C:\Users\josen\OneDrive\Desktop\Work\Post-doc_Recife\alternative_projects\Rabello's_project\CV_dropbox_content\Projeto_Ze_Gabrielzinho_Rabelo_Planeta\Jupiter\cycle_npa_results_tight_scs


## 1. Canonical coordinates and the corrected cycle polytope

The full behavior is a 64-vector.  The input blocks are ordered

`xyz = 000, 001, 010, 011, 100, 101, 110, 111`,

and inside each block the outputs are ordered

`abc = 000, 001, 010, 011, 100, 101, 110, 111`.

Thus `c` and `z` are the fastest-changing indices.  The corrected deterministic
dependencies are

- clockwise: \(a=f_A(x,z),\ b=f_B(y,x),\ c=f_C(z,y)\);
- counterclockwise: \(a=f_A(x,y),\ b=f_B(y,z),\ c=f_C(z,x)\).

Each orientation has 4096 vertices.  Their intersection contains the 64 local
deterministic behaviors, so their distinct union contains 8128 vertices.


In [6]:
"""
    prob_index(a,b,c,x,y,z)

Return the one-based position of p(a,b,c|x,y,z) in the canonical 64-vector.
"""
prob_index(a, b, c, x, y, z) =
    1 + c + 2*b + 4*a + 8*z + 16*y + 32*x

@assert [prob_index(a, b, c, 0, 0, 0) for (a, b, c) in TRIPLES] == collect(1:8)
@assert [prob_index(0, 0, 0, x, y, z) for (x, y, z) in TRIPLES] == collect(1:8:57)

"""Evaluate one of the 16 deterministic functions of two binary inputs."""
function response(mask::Int, u::Int, v::Int)
    @assert 0 <= mask <= 15
    @assert u in 0:1 && v in 0:1
    return Int((mask >> (2*u + v)) & 1)
end

"""Construct one 64-component deterministic vertex of one cycle orientation."""
function deterministic_vertex(fA::Int, fB::Int, fC::Int, orientation::Symbol)
    vertex = zeros(Int8, 64)

    for (x, y, z) in TRIPLES
        if orientation === :clockwise
            a = response(fA, x, z)
            b = response(fB, y, x)
            c = response(fC, z, y)
        elseif orientation === :counterclockwise
            a = response(fA, x, y)
            b = response(fB, y, z)
            c = response(fC, z, x)
        else
            error("Unknown cycle orientation: $orientation")
        end

        vertex[prob_index(a, b, c, x, y, z)] = 1
    end

    return vertex
end

"""Generate all 16^3 deterministic vertices of one orientation."""
function generate_orientation(orientation::Symbol)
    vertices = Matrix{Int8}(undef, 16^3, 64)
    row = 1

    for fA in 0:15, fB in 0:15, fC in 0:15
        vertices[row, :] .= deterministic_vertex(fA, fB, fC, orientation)
        row += 1
    end

    return vertices
end

"""Generate the 4^3 ordinary local deterministic behaviors."""
function generate_local_vertices()
    vertices = Matrix{Int8}(undef, 4^3, 64)
    row = 1

    for fA in 0:3, fB in 0:3, fC in 0:3
        vertex = zeros(Int8, 64)
        for (x, y, z) in TRIPLES
            a = Int((fA >> x) & 1)
            b = Int((fB >> y) & 1)
            c = Int((fC >> z) & 1)
            vertex[prob_index(a, b, c, x, y, z)] = 1
        end
        vertices[row, :] .= vertex
        row += 1
    end

    return vertices
end

row_set(vertices) = Set(Tuple(row) for row in eachrow(vertices))

function affine_dimension(points; atol=1e-9)
    n_points = size(points, 1)
    n_points == 0 && return -1
    n_points == 1 && return 0

    differences = Float64.(points[2:end, :]) .- Float64.(points[1:1, :])
    return rank(differences; atol=atol, rtol=0.0)
end

function validate_deterministic_vertices(vertices)
    @assert size(vertices, 2) == 64
    for vertex in eachrow(vertices)
        @assert all(value -> value == 0 || value == 1, vertex)
        for context in 0:7
            @assert sum(vertex[(8*context + 1):(8*context + 8)]) == 1
        end
    end
    return true
end


validate_deterministic_vertices (generic function with 1 method)

In [7]:
vertices_clockwise = generate_orientation(:clockwise)
vertices_counterclockwise = generate_orientation(:counterclockwise)
vertices_cycle = unique(
    vcat(vertices_clockwise, vertices_counterclockwise);
    dims=1,
)
vertices_local = generate_local_vertices()

clockwise_set = row_set(vertices_clockwise)
counterclockwise_set = row_set(vertices_counterclockwise)
local_set = row_set(vertices_local)

validate_deterministic_vertices(vertices_clockwise)
validate_deterministic_vertices(vertices_counterclockwise)
validate_deterministic_vertices(vertices_cycle)
validate_deterministic_vertices(vertices_local)

@assert size(vertices_clockwise) == (4096, 64)
@assert size(vertices_counterclockwise) == (4096, 64)
@assert length(intersect(clockwise_set, counterclockwise_set)) == 64
@assert intersect(clockwise_set, counterclockwise_set) == local_set
@assert size(vertices_cycle) == (8128, 64)
@assert affine_dimension(vertices_cycle) == 50

println("Corrected cycle polytope reconstructed successfully")
println("  clockwise vertices:        ", size(vertices_clockwise, 1))
println("  counterclockwise vertices: ", size(vertices_counterclockwise, 1))
println("  local intersection:        ", size(vertices_local, 1))
println("  distinct union:            ", size(vertices_cycle, 1))
println("  affine dimension:          ", affine_dimension(vertices_cycle))


Corrected cycle polytope reconstructed successfully
  clockwise vertices:        4096
  counterclockwise vertices: 4096
  local intersection:        64
  distinct union:            8128
  affine dimension:          50


## 2. The four nonsignalling targets and their initial separators

The representatives are entered through their 26 correlators in the order

`A0,A1, B0,B1, C0,C1, AB00,...,AB11, AC00,...,AC11, BC00,...,BC11, ABC000,...,ABC111`.

They are converted exactly to the 64 probabilities.  For each target \(q\) we
then solve

\[
\min_c\ c\cdot q \quad\text{subject to}\quad
c\cdot v\ge1\;\text{for every cycle vertex }v,\qquad c\ge0.
\]

These four inequalities are the outputs of stage 2.  They are included even
when the perturbation search happened to rediscover the same coefficient
vector.


In [9]:
NS_CORRELATORS = Dict(
    27 => Rational{Int}[
        1//2, 0, 0, 0, 0, 0,
        1//2, 1//2, 0, 0,
        1//2, 1//2, 0, 0,
        0, 0, 0, 0,
        -1//2, -1//2, -1//2, -1//2, 1, 1, 1, -1,
    ],
    29 => Rational{Int}[
        1//3, 1//3, 1//3, 1//3, 1//3, 1//3,
        1, -1//3, -1//3, -1//3,
        1, -1//3, -1//3, -1//3,
        1, -1//3, -1//3, -1//3,
        1//3, 1//3, 1//3, -1, 1//3, -1, -1, -1,
    ],
    43 => Rational{Int}[
        1//7, 1//7, 1//7, 1//7, 1//7, 1//7,
        3//7, -1//7, -1//7, -1//7,
        3//7, -1//7, -1//7, -1//7,
        -1//7, -1//7, -1//7, -1//7,
        1//7, 5//7, 5//7, -1, -1, -1, -1, -1,
    ],
    44 => Rational{Int}[
        0, 0, 0, 0, 0, 0,
        0, 0, 0, 0,
        0, 0, 0, 0,
        0, 0, 0, 0,
        1, 1, 1, 1, 1, 1, 1, -1,
    ],
)

outcome_sign(bit::Int) = bit == 0 ? 1 : -1

"""Convert the 26 correlators to canonical full probabilities exactly."""
function correlators_to_probabilities(correlators)
    @assert length(correlators) == 26
    q = Rational{Int}.(correlators)
    probabilities = fill(0//1, 64)

    A = q[1:2]
    B = q[3:4]
    C = q[5:6]
    AB = q[7:10]
    AC = q[11:14]
    BC = q[15:18]
    ABC = q[19:26]

    for (x, y, z) in TRIPLES, (a, b, c) in TRIPLES
        sa = outcome_sign(a)
        sb = outcome_sign(b)
        sc = outcome_sign(c)

        probabilities[prob_index(a, b, c, x, y, z)] = (
            1 +
            sa*A[x + 1] +
            sb*B[y + 1] +
            sc*C[z + 1] +
            sa*sb*AB[2*x + y + 1] +
            sa*sc*AC[2*x + z + 1] +
            sb*sc*BC[2*y + z + 1] +
            sa*sb*sc*ABC[4*x + 2*y + z + 1]
        ) / 8
    end

    return probabilities
end

NS_TARGETS = Dict(
    class => correlators_to_probabilities(NS_CORRELATORS[class])
    for class in CLASSES
)

function validate_nonsignalling_behavior(probabilities)
    @assert length(probabilities) == 64
    @assert all(probability -> probability >= 0, probabilities)

    for (x, y, z) in TRIPLES
        @assert sum(
            probabilities[prob_index(a, b, c, x, y, z)]
            for (a, b, c) in TRIPLES
        ) == 1
    end

    # AB marginal independent of z.
    for x in 0:1, y in 0:1, a in 0:1, b in 0:1
        lhs = sum(probabilities[prob_index(a, b, c, x, y, 0)] for c in 0:1)
        rhs = sum(probabilities[prob_index(a, b, c, x, y, 1)] for c in 0:1)
        @assert lhs == rhs
    end

    # AC marginal independent of y.
    for x in 0:1, z in 0:1, a in 0:1, c in 0:1
        lhs = sum(probabilities[prob_index(a, b, c, x, 0, z)] for b in 0:1)
        rhs = sum(probabilities[prob_index(a, b, c, x, 1, z)] for b in 0:1)
        @assert lhs == rhs
    end

    # BC marginal independent of x.
    for y in 0:1, z in 0:1, b in 0:1, c in 0:1
        lhs = sum(probabilities[prob_index(a, b, c, 0, y, z)] for a in 0:1)
        rhs = sum(probabilities[prob_index(a, b, c, 1, y, z)] for a in 0:1)
        @assert lhs == rhs
    end

    return true
end

for class in CLASSES
    validate_nonsignalling_behavior(NS_TARGETS[class])
end

println("All four NS representatives passed exact positivity, normalization, and nonsignalling checks.")


All four NS representatives passed exact positivity, normalization, and nonsignalling checks.


In [10]:
"""Evaluate an exact rational coefficient vector on deterministic vertices."""
function exact_vertex_values(vertices, coefficients)
    n_vertices, dimension = size(vertices)
    @assert dimension == length(coefficients) == 64

    values = Vector{Rational{BigInt}}(undef, n_vertices)
    zero_rational = BigInt(0) // BigInt(1)

    @inbounds for row in 1:n_vertices
        value = zero_rational
        for column in 1:dimension
            if vertices[row, column] != 0
                value += coefficients[column]
            end
        end
        values[row] = value
    end

    return values
end

function rationalize_coefficients(coefficients; tolerance=1e-8)
    return Rational{BigInt}[
        abs(value) <= tolerance ?
            BigInt(0)//BigInt(1) :
            rationalize(BigInt, max(value, 0.0); tol=tolerance)
        for value in coefficients
    ]
end

"""
Solve the initial separating LP and turn its floating-point output into a
bound-normalized exact rational inequality.
"""
function solve_initial_separator(class, target, vertices; tolerance=1e-8)
    n_vertices, dimension = size(vertices)
    model = Model(GLPK.Optimizer)
    set_silent(model)

    @variable(model, coefficients[1:dimension] >= 0)
    @objective(
        model,
        Min,
        sum(Float64(target[j]) * coefficients[j] for j in 1:dimension),
    )
    @constraint(
        model,
        classical[i=1:n_vertices],
        sum(vertices[i, j] * coefficients[j] for j in 1:dimension) >= 1,
    )

    optimize!(model)
    status = termination_status(model)
    status == MOI.OPTIMAL || error(
        "Initial LP for class $class ended with status $status",
    )

    raw_exact = rationalize_coefficients(value.(coefficients); tolerance=tolerance)
    raw_values = exact_vertex_values(vertices, raw_exact)
    raw_minimum = minimum(raw_values)
    raw_minimum > 0 || error("Initial separator for class $class has nonpositive bound")

    exact_coefficients = raw_exact ./ raw_minimum
    exact_values = raw_values ./ raw_minimum
    @assert minimum(exact_values) == 1

    target_value = sum(exact_coefficients .* Rational{BigInt}.(target))
    saturating = findall(==(BigInt(1)//BigInt(1)), exact_values)
    face_dimension = affine_dimension(vertices[saturating, :])

    return (
        discovery_class=class,
        source="initial-LP",
        pair="",
        coefficients=exact_coefficients,
        target_value=target_value,
        saturation_count=length(saturating),
        face_dimension=face_dimension,
        is_facet=(face_dimension == 49),
    )
end

initial_records = [
    solve_initial_separator(class, NS_TARGETS[class], vertices_cycle)
    for class in CLASSES
]

expected_initial_values = Dict(
    27 => BigInt(3)//BigInt(4),
    29 => BigInt(1)//BigInt(2),
    43 => BigInt(5)//BigInt(7),
    44 => BigInt(0)//BigInt(1),
)

for record in initial_records
    @assert record.target_value == expected_initial_values[record.discovery_class]
end

println("Initial LP separators regenerated")
println("class  target value  saturation  face dimension  facet")
for record in initial_records
    @printf(
        "%5d  %12s  %10d  %14d  %s\n",
        record.discovery_class,
        string(record.target_value),
        record.saturation_count,
        record.face_dimension,
        string(record.is_facet),
    )
end


Initial LP separators regenerated
class  target value  saturation  face dimension  facet
   27          3//4         119              49  true
   29          1//2          90              44  false
   43          5//7         508              49  true
   44          0//1         508              49  true


## 3. Read the perturbation-search inequalities

The perturbation notebook already saved the coefficients as exact rational
numbers.  This cell reads those exact strings directly; it never rounds them
back through floating point.

The four initial separators and all perturbation records are then merged by
their complete 64-component coefficient vector.  We deliberately do **not**
identify two different vectors merely because they expose the same face.


In [12]:
function parse_big_rational(text::AbstractString)
    cleaned = strip(text)
    if occursin('/', cleaned)
        pieces = split(cleaned, '/'; limit=2)
        length(pieces) == 2 || error("Invalid rational: $text")
        denominator_value = parse(BigInt, pieces[2])
        denominator_value != 0 || error("Zero denominator in: $text")
        return parse(BigInt, pieces[1]) // denominator_value
    end
    return parse(BigInt, cleaned) // BigInt(1)
end

function rational_string(value::Rational)
    denominator(value) == 1 && return string(numerator(value))
    return "$(numerator(value))/$(denominator(value))"
end

coefficient_key(coefficients) = join(rational_string.(coefficients), ",")

function read_perturbation_table(path)
    isfile(path) || error(
        "Missing perturbation table: $path\n" *
        "Place this notebook beside the cycle_perturbation_results directory " *
        "or edit PERTURBATION_RESULTS_DIRECTORY in the configuration cell.",
    )

    lines = readlines(path)
    isempty(lines) && error("Empty perturbation table: $path")

    header = split(chomp(lines[1]), '\t')
    column = Dict(name => index for (index, name) in enumerate(header))
    required = [
        "class",
        "first_j",
        "first_k",
        "coefficients_in_canonical_order",
    ]
    all(name -> haskey(column, name), required) || error(
        "Unexpected columns in perturbation table: $path",
    )

    records = Any[]
    for line in lines[2:end]
        isempty(strip(line)) && continue
        fields = split(chomp(line), '\t'; keepempty=true)
        length(fields) == length(header) || error("Malformed TSV row in $path")

        class = parse(Int, fields[column["class"]])
        first_j = parse(Int, fields[column["first_j"]])
        first_k = parse(Int, fields[column["first_k"]])
        coefficients = parse_big_rational.(split(
            fields[column["coefficients_in_canonical_order"]],
            ',',
        ))
        length(coefficients) == 64 || error(
            "Candidate from class $class, pair ($first_j,$first_k) has " *
            "$(length(coefficients)) coefficients instead of 64.",
        )

        push!(records, (
            discovery_class=class,
            source="perturbation",
            pair="C$class:($first_j,$first_k)",
            coefficients=coefficients,
        ))
    end

    return records
end

perturbation_records = Any[]
for class in CLASSES
    path = joinpath(
        PERTURBATION_RESULTS_DIRECTORY,
        "class_$(lpad(string(class), 2, '0'))_candidates.tsv",
    )
    append!(perturbation_records, read_perturbation_table(path))
end

println("Loaded $(length(perturbation_records)) perturbation-search records.")


Loaded 20 perturbation-search records.


In [13]:
"""Merge provenance while deduplicating only complete exact coefficient vectors."""
function combine_exact_candidates(initial_records, perturbation_records)
    grouped = Dict{String, Dict{String, Any}}()
    key_order = String[]

    for record in vcat(initial_records, perturbation_records)
        key = coefficient_key(record.coefficients)
        if !haskey(grouped, key)
            grouped[key] = Dict{String, Any}(
                "coefficients" => Rational{BigInt}.(record.coefficients),
                "classes" => Set{Int}(),
                "sources" => Set{String}(),
                "pairs" => String[],
            )
            push!(key_order, key)
        end

        group = grouped[key]
        push!(group["classes"], record.discovery_class)
        push!(group["sources"], record.source)
        if !isempty(record.pair) && !(record.pair in group["pairs"])
            push!(group["pairs"], record.pair)
        end
    end

    candidates = Any[]
    for (index, key) in enumerate(key_order)
        group = grouped[key]
        push!(candidates, (
            candidate_id=@sprintf("W%03d", index),
            key=key,
            coefficients=group["coefficients"],
            discovery_classes=sort!(collect(group["classes"])),
            sources=sort!(collect(group["sources"])),
            perturbation_pairs=copy(group["pairs"]),
        ))
    end

    return candidates
end

candidates = combine_exact_candidates(initial_records, perturbation_records)
@assert length(candidates) >= length(CLASSES)

println(
    "Union of stage 2 and stage 4: $(length(candidates)) distinct exact " *
    "coefficient vectors.",
)


Union of stage 2 and stage 4: 20 distinct exact coefficient vectors.


## 4. Exact classical audit before NPA

Every candidate is evaluated exactly on all 8128 corrected cycle vertices.
The notebook stops if its minimum is not exactly `1`.  It also records the
minimum over the 64 ordinary local vertices, the number of saturating cycle
vertices, and the face dimension.

This step prevents a coordinate-order mistake or a malformed input row from
being passed silently to the quantum calculation.


In [15]:
function audit_classical_candidate(candidate, vertices_cycle, vertices_local)
    cycle_values = exact_vertex_values(vertices_cycle, candidate.coefficients)
    cycle_bound = minimum(cycle_values)
    cycle_bound == 1 || error(
        "Candidate $(candidate.candidate_id) has exact cycle bound " *
        "$cycle_bound instead of 1.",
    )

    saturating = findall(==(BigInt(1)//BigInt(1)), cycle_values)
    face_dimension = affine_dimension(vertices_cycle[saturating, :])

    local_values = exact_vertex_values(vertices_local, candidate.coefficients)
    local_bound = minimum(local_values)

    return merge(candidate, (
        cycle_bound=cycle_bound,
        local_bound=local_bound,
        saturation_count=length(saturating),
        face_dimension=face_dimension,
        is_facet=(face_dimension == 49),
    ))
end

audited_candidates = [
    audit_classical_candidate(candidate, vertices_cycle, vertices_local)
    for candidate in candidates
]

println("Exact classical audit passed for all $(length(audited_candidates)) candidates.")
println("ID    classes       sources                    local  sat.  dim  facet")
for candidate in audited_candidates
    @printf(
        "%-5s %-13s %-26s %-6s %5d %4d  %s\n",
        candidate.candidate_id,
        join(candidate.discovery_classes, ","),
        join(candidate.sources, "+"),
        rational_string(candidate.local_bound),
        candidate.saturation_count,
        candidate.face_dimension,
        string(candidate.is_facet),
    )
end


Exact classical audit passed for all 20 candidates.
ID    classes       sources                    local  sat.  dim  facet
W001  27            initial-LP+perturbation    1        119   49  true
W002  29            initial-LP+perturbation    1         90   44  false
W003  43            initial-LP+perturbation    1        508   49  true
W004  44            initial-LP+perturbation    1        508   49  true
W005  27            perturbation               1        176   49  true
W006  27            perturbation               1        119   49  true
W007  27            perturbation               1        177   49  true
W008  27            perturbation               1        176   49  true
W009  27            perturbation               1        176   49  true
W010  29            perturbation               1         96   45  false
W011  29            perturbation               1         84   47  false
W012  29            perturbation               1         80   46  false
W013  29            p

## 5. Automatic Bell-operator construction

For Alice, Bob, and Charlie we create the two outcomes of each of two
measurements with `projector(..., full=true)`.  Hence, for example, the second
outcome is represented internally as identity minus the first outcome.  The
operator corresponding to a probability is

\[
P^A_{a\mid x}P^B_{b\mid y}P^C_{c\mid z}.
\]

The same `prob_index` function used for the classical vectors places every
coefficient in front of the correct probability operator.  No 8×8 matrix is
transcribed manually.

The small compatibility method below is the same issue already encountered in
Carlos's notebook: some combinations of `QuantumNPA`, `JuMP`, and recent Julia
versions lack the sparse-matrix/`VariableRef` dot-product specialization.  It
is added only when that method is absent.


In [17]:
function install_quantumnpa_jump_compatibility!()
    signature = Tuple{
        SparseMatrixCSC{Float64, Int},
        Symmetric{JuMP.VariableRef, Matrix{JuMP.VariableRef}},
    }

    if !isdefined(QuantumNPA, :dot) ||
       !hasmethod(getfield(QuantumNPA, :dot), signature)
        @eval QuantumNPA begin
            import JuMP: VariableRef
            import LinearAlgebra: Symmetric
            import SparseArrays: SparseMatrixCSC

            function dot(
                A::SparseMatrixCSC{Float64, Int},
                B::Symmetric{VariableRef, Matrix{VariableRef}},
            )
                return sum(
                    A[i, j] * B[j, i]
                    for i in 1:size(A, 1), j in 1:size(A, 2)
                )
            end
        end
        println("Installed the QuantumNPA/JuMP sparse-dot compatibility method.")
    else
        println("QuantumNPA/JuMP sparse-dot compatibility method already available.")
    end
end

install_quantumnpa_jump_compatibility!()

"""Create the 64 probability operators in exactly the canonical vector order."""
function build_probability_operators()
    PA = projector(1, 1:2, 1:2, full=true)
    PB = projector(2, 1:2, 1:2, full=true)
    PC = projector(3, 1:2, 1:2, full=true)

    operators = Vector{Any}(undef, 64)
    for (x, y, z) in TRIPLES, (a, b, c) in TRIPLES
        operators[prob_index(a, b, c, x, y, z)] =
            PA[a + 1, x + 1] * PB[b + 1, y + 1] * PC[c + 1, z + 1]
    end
    return operators
end

PROBABILITY_OPERATORS = build_probability_operators()
@assert length(PROBABILITY_OPERATORS) == 64

"""Build one QuantumNPA polynomial from an exact 64-coefficient vector."""
function build_bell_expression(coefficients, probability_operators)
    length(coefficients) == length(probability_operators) == 64 ||
        error("A Bell expression requires exactly 64 coefficients/operators")

    expression = nothing
    for index in eachindex(coefficients)
        iszero(coefficients[index]) && continue
        term = Float64(coefficients[index]) * probability_operators[index]
        expression = expression === nothing ? term : expression + term
    end

    expression === nothing && error("The Bell functional is identically zero")
    return expression
end

# Construct every expression once now.  This is a structural smoke test before
# the first SDP is launched.
for candidate in audited_candidates
    build_bell_expression(candidate.coefficients, PROBABILITY_OPERATORS)
end

println("All Bell expressions were constructed in canonical p(abc|xyz) order.")


QuantumNPA/JuMP sparse-dot compatibility method already available.
All Bell expressions were constructed in canonical p(abc|xyz) order.


## 6. Restartable NPA scan

For each exact inequality the notebook computes `npa_min` at levels 2, 3, and
4 with SCS.  After every successful SDP it rewrites the human-readable tables
and a binary checkpoint.

For an inequality with cycle bound \(\beta_C=1\), the reported quantities are

- `ratio_level4 = L4 / beta_C = L4`;
- `normalized_gap = 1 - ratio_level4`;
- `inverse_ratio = beta_C / L4` when `L4 > 0`.

Here `L4` is the level-4 NPA lower bound, not yet the value of an explicit
quantum strategy.  Smaller `ratio_level4` (equivalently, larger positive gap)
is the ranking convention proposed by Carlos.


In [19]:
function npa_paths(output_directory)
    return (
        checkpoint=joinpath(output_directory, "quantum_npa_checkpoint.jls"),
        table=joinpath(output_directory, "quantum_npa_results.tsv"),
        promising=joinpath(output_directory, "quantum_npa_promising.tsv"),
        metadata=joinpath(output_directory, "run_metadata.txt"),
    )
end

function write_checkpoint(path, state)
    temporary = path * ".tmp"
    open(temporary, "w") do io
        serialize(io, state)
    end
    mv(temporary, path; force=true)
    return path
end

function new_npa_state(candidates, levels)
    return Dict{String, Any}(
        "format_version" => 1,
        "candidate_keys" => [candidate.key for candidate in candidates],
        "levels" => collect(levels),
        "values" => Dict{String, Dict{Int, Float64}}(),
        "seconds" => Dict{String, Dict{Int, Float64}}(),
        "errors" => Dict{String, Dict{Int, String}}(),
        "updated_at" => string(now()),
    )
end

function load_or_create_npa_state(path, candidates, levels; resume=true)
    if resume && isfile(path)
        state = open(path, "r") do io
            deserialize(io)
        end
        state["format_version"] == 1 || error("Unknown NPA checkpoint format")
        state["candidate_keys"] == [candidate.key for candidate in candidates] ||
            error(
                "The candidate set differs from the existing NPA checkpoint. " *
                "Set RESUME_NPA_SCAN=false to start a new scan.",
            )
        state["levels"] == collect(levels) || error(
            "NPA_LEVELS differs from the existing checkpoint. " *
            "Set RESUME_NPA_SCAN=false to start a new scan.",
        )
        println("Resuming NPA scan from: ", abspath(path))
        return state
    end

    println("Starting a new NPA scan.")
    return new_npa_state(candidates, levels)
end

function value_at(state, candidate, level)
    values = get(state["values"], candidate.key, nothing)
    values === nothing && return missing
    return get(values, level, missing)
end

function level4_label(value, cycle_bound; tolerance)
    ismissing(value) && return "incomplete"
    ratio = value / Float64(cycle_bound)
    if ratio < 1 - tolerance
        return "promising_level4"
    elseif ratio <= 1 + tolerance
        return "no_resolved_level4_gap"
    end
    return "no_level4_violation"
end

function monotonicity_ok(state, candidate, levels; tolerance)
    values = [value_at(state, candidate, level) for level in levels]
    any(ismissing, values) && return missing
    return all(values[i] <= values[i + 1] + tolerance for i in 1:(length(values) - 1))
end

function result_rows(candidates, state, levels; screening_tolerance, monotonicity_tolerance)
    rows = Any[]
    for candidate in candidates
        level_values = Dict(level => value_at(state, candidate, level) for level in levels)
        level4 = get(level_values, 4, missing)
        ratio = ismissing(level4) ? missing : level4 / Float64(candidate.cycle_bound)
        gap = ismissing(ratio) ? missing : 1 - ratio
        inverse = ismissing(ratio) || ratio <= 0 ? missing : 1 / ratio

        push!(rows, (
            candidate=candidate,
            values=level_values,
            ratio=ratio,
            gap=gap,
            inverse=inverse,
            label=level4_label(
                level4,
                candidate.cycle_bound;
                tolerance=screening_tolerance,
            ),
            monotonic=monotonicity_ok(
                state,
                candidate,
                levels;
                tolerance=monotonicity_tolerance,
            ),
        ))
    end

    sort!(rows; by=row -> ismissing(row.ratio) ? Inf : row.ratio)
    return rows
end

text_value(value::Missing) = ""
text_value(value::Bool) = string(value)
text_value(value::Real) = repr(Float64(value))
text_value(value) = string(value)

function write_npa_table(path, rows, levels)
    header = [
        "rank_by_level4",
        "candidate_id",
        "discovery_classes",
        "sources",
        "perturbation_pairs",
        "cycle_bound_exact",
        "local_bound_exact",
        "saturation_count",
        "face_dimension",
        "is_facet",
    ]
    append!(header, ["npa_level_$level" for level in levels])
    append!(header, [
        "ratio_level4",
        "normalized_gap_1_minus_ratio",
        "inverse_ratio",
        "npa_monotonicity_ok",
        "screening_label",
        "coefficients_in_canonical_order",
    ])

    open(path, "w") do io
        println(io, join(header, '\t'))
        for (rank_index, row) in enumerate(rows)
            candidate = row.candidate
            fields = String[
                string(rank_index),
                candidate.candidate_id,
                join(candidate.discovery_classes, ","),
                join(candidate.sources, "+"),
                join(candidate.perturbation_pairs, ";"),
                rational_string(candidate.cycle_bound),
                rational_string(candidate.local_bound),
                string(candidate.saturation_count),
                string(candidate.face_dimension),
                string(candidate.is_facet),
            ]
            append!(fields, [text_value(row.values[level]) for level in levels])
            append!(fields, [
                text_value(row.ratio),
                text_value(row.gap),
                text_value(row.inverse),
                text_value(row.monotonic),
                row.label,
                coefficient_key(candidate.coefficients),
            ])
            println(io, join(fields, '\t'))
        end
    end
    return path
end

function write_npa_tables(paths, candidates, state, levels; screening_tolerance, monotonicity_tolerance)
    rows = result_rows(
        candidates,
        state,
        levels;
        screening_tolerance=screening_tolerance,
        monotonicity_tolerance=monotonicity_tolerance,
    )
    write_npa_table(paths.table, rows, levels)

    promising_rows = [row for row in rows if row.label == "promising_level4"]
    write_npa_table(paths.promising, promising_rows, levels)
    return rows
end


write_npa_tables (generic function with 1 method)

In [20]:
function run_npa_scan!(
    candidates,
    probability_operators;
    levels,
    output_directory,
    resume,
    stop_on_error,
    screening_tolerance,
    monotonicity_tolerance,
)
    mkpath(output_directory)
    paths = npa_paths(output_directory)
    state = load_or_create_npa_state(paths.checkpoint, candidates, levels; resume=resume)

    for (candidate_index, candidate) in enumerate(candidates)
        candidate_values = get!(
            state["values"],
            candidate.key,
            Dict{Int, Float64}(),
        )
        candidate_seconds = get!(
            state["seconds"],
            candidate.key,
            Dict{Int, Float64}(),
        )
        candidate_errors = get!(
            state["errors"],
            candidate.key,
            Dict{Int, String}(),
        )

        pending_levels = [level for level in levels if !haskey(candidate_values, level)]
        if isempty(pending_levels)
            println(
                "[$candidate_index/$(length(candidates))] " *
                "$(candidate.candidate_id): already complete",
            )
            continue
        end

        expression = build_bell_expression(
            candidate.coefficients,
            probability_operators,
        )

        for level in pending_levels
            println(
                "[$candidate_index/$(length(candidates))] " *
                "$(candidate.candidate_id): starting NPA level $level",
            )
            started_at = time()

            try
                # Use the explicitly configured tighter SCS optimizer instead
                # of QuantumNPA's default solver settings.
                bound = npa_min(expression, level; solver=TIGHT_SCS_SOLVER)
                isfinite(bound) || error("QuantumNPA returned non-finite value $bound")

                candidate_values[level] = Float64(bound)
                candidate_seconds[level] = time() - started_at
                delete!(candidate_errors, level)
                state["updated_at"] = string(now())

                write_checkpoint(paths.checkpoint, state)
                write_npa_tables(
                    paths,
                    candidates,
                    state,
                    levels;
                    screening_tolerance=screening_tolerance,
                    monotonicity_tolerance=monotonicity_tolerance,
                )

                @printf(
                    "    value = %.12g, elapsed = %.1f s\n",
                    bound,
                    candidate_seconds[level],
                )
            catch error_value
                candidate_errors[level] = sprint(showerror, error_value)
                state["updated_at"] = string(now())
                write_checkpoint(paths.checkpoint, state)
                write_npa_tables(
                    paths,
                    candidates,
                    state,
                    levels;
                    screening_tolerance=screening_tolerance,
                    monotonicity_tolerance=monotonicity_tolerance,
                )

                @error "NPA calculation failed" candidate=candidate.candidate_id level=level exception=(error_value, catch_backtrace())
                stop_on_error && rethrow()
            end
        end

        # Level-4 moment matrices are the largest objects in this run.  Let
        # Julia reclaim them before constructing the next candidate.
        expression = nothing
        GC.gc()
    end

    rows = write_npa_tables(
        paths,
        candidates,
        state,
        levels;
        screening_tolerance=screening_tolerance,
        monotonicity_tolerance=monotonicity_tolerance,
    )
    write_checkpoint(paths.checkpoint, state)
    return state, rows, paths
end

npa_state, ranked_rows, output_paths = run_npa_scan!(
    audited_candidates,
    PROBABILITY_OPERATORS;
    levels=NPA_LEVELS,
    output_directory=NPA_RESULTS_DIRECTORY,
    resume=RESUME_NPA_SCAN,
    stop_on_error=STOP_ON_NPA_ERROR,
    screening_tolerance=NPA_SCREENING_TOLERANCE,
    monotonicity_tolerance=NPA_MONOTONICITY_TOLERANCE,
)


Starting a new NPA scan.
[1/20] W001: starting NPA level 2
    value = 1.00000000009, elapsed = 17.8 s
[1/20] W001: starting NPA level 3
    value = 1.00000004485, elapsed = 0.5 s
[1/20] W001: starting NPA level 4
    value = 1.00000000004, elapsed = 2.5 s
[2/20] W002: starting NPA level 2
    value = 1.00000000028, elapsed = 0.1 s
[2/20] W002: starting NPA level 3
    value = 1.00000000038, elapsed = 0.8 s
[2/20] W002: starting NPA level 4
    value = 0.999999999912, elapsed = 3.7 s
[3/20] W003: starting NPA level 2
    value = 1.00000001291, elapsed = 5.2 s
[3/20] W003: starting NPA level 3
    value = 0.999999989909, elapsed = 7.9 s
[3/20] W003: starting NPA level 4
    value = 0.999999986768, elapsed = 187.9 s
[4/20] W004: starting NPA level 2
    value = 1.0000000824, elapsed = 9.0 s
[4/20] W004: starting NPA level 3
    value = 1.00000000318, elapsed = 11.2 s
[4/20] W004: starting NPA level 4
    value = 1.00000006802, elapsed = 13.0 s
[5/20] W005: starting NPA level 2
    value 

(Dict{String, Any}("format_version" => 1, "levels" => [2, 3, 4], "errors" => Dict("0,1,1/2,1,1,1,1,0,0,1/2,1,1,1,1,0,1/2,0,1,1/2,1,1,0,1,0,1/2,0,0,0,0,1,1,1,0,1,1,0,1/2,1,1,1/2,1/2,0,0,1,0,0,1,1/2,1/2,0,0,1,0,1,0,1,1,0,0,1/2,0,1/2,1/2,1" => Dict(), "0,1,1,1,1,1,1,0,0,1/2,1,1,1,1,0,0,0,1,1/2,1,1,0,1,0,1/2,0,0,0,0,1,1,1,0,1,1,0,1/2,1,1,0,1/2,0,0,1,0,0,1,1,1/2,0,0,1,0,1,0,1,1,0,0,1/2,0,1/2,1/2,1" => Dict(), "1,1,1,1,1,1,1,0,0,0,0,1,1,1,1,0,0,0,0,1,1,1,1,0,0,0,0,1,0,1,1,0,0,1,1,0,0,0,0,1,0,1,1,0,1,0,0,1,0,1,1,0,1,0,0,1,1,0,0,1,0,1,1,0" => Dict(), "0,1,1,1,1/2,1,1,0,0,1/2,1,1,1,1,0,1,0,1,1/2,1,1,0,1,1,1/2,0,0,0,0,1,1,0,0,1,1,0,1/2,1,1,0,1/2,0,0,1,0,0,1,1,1/2,0,0,1,0,1,0,1,1,0,0,1/2,0,1/2,1/2,1" => Dict(), "0,1/2,1/2,1,1/2,1,1,0,0,1/2,1,1,1,1,0,1,0,1,1/2,1,1,0,1,1,1/2,0,0,0,0,1,1,0,0,1,1,0,1/2,1,1,1/2,1/2,0,0,1,0,0,1,1/2,1/2,0,0,1,0,1,0,1/2,1,0,0,1/2,0,1/2,1/2,1" => Dict(), "0,1/2,1/2,1,1/2,1,1,0,0,1/2,1,1,1,1,0,1,0,1,1/2,1,1,0,1,1,1/2,0,0,0,0,1,1,0,0,1,1,0,1/2,1,1,1,1/2,0,0,1,0,0,1,0,1/2,0,

## 7. Ranked summary and numerical diagnostics

In the table below, a smaller level-4 ratio is the more promising result.
`monotone` checks the theoretical ordering \(L_2\le L_3\le L_4\) up to the
configured numerical tolerance.  We also verify that each NPA lower bound does
not exceed the exact local deterministic value by more than that tolerance.


In [22]:
println("\nRanked level-4 summary")
println("rank  ID    classes       L2            L3            L4            gap           label")

for (rank_index, row) in enumerate(ranked_rows)
    candidate = row.candidate
    l2 = get(row.values, 2, missing)
    l3 = get(row.values, 3, missing)
    l4 = get(row.values, 4, missing)

    @printf(
        "%4d  %-5s %-13s %-13s %-13s %-13s %-13s %s\n",
        rank_index,
        candidate.candidate_id,
        join(candidate.discovery_classes, ","),
        ismissing(l2) ? "—" : @sprintf("%.9f", l2),
        ismissing(l3) ? "—" : @sprintf("%.9f", l3),
        ismissing(l4) ? "—" : @sprintf("%.9f", l4),
        ismissing(row.gap) ? "—" : @sprintf("%.9f", row.gap),
        row.label,
    )

    if row.monotonic === false
        @warn "NPA levels are not numerically monotone within tolerance" candidate=candidate.candidate_id level2=l2 level3=l3 level4=l4
    end

    if !ismissing(l4) &&
       l4 > Float64(candidate.local_bound) + NPA_MONOTONICITY_TOLERANCE
        @warn "Level-4 lower bound exceeds the exact local value" candidate=candidate.candidate_id level4=l4 local_bound=candidate.local_bound
    end
end

promising_count = count(row -> row.label == "promising_level4", ranked_rows)
println("\nPromising level-4 candidates: $promising_count")
println("Full table:      ", abspath(output_paths.table))
println("Promising table: ", abspath(output_paths.promising))
println("Checkpoint:      ", abspath(output_paths.checkpoint))



Ranked level-4 summary
rank  ID    classes       L2            L3            L4            gap           label
   1  W010  29            1.000000002   1.000000000   0.999999984   0.000000016   no_resolved_level4_gap
   2  W003  43            1.000000013   0.999999990   0.999999987   0.000000013   no_resolved_level4_gap
   3  W011  29            0.996403590   1.000000000   0.999999992   0.000000008   no_resolved_level4_gap
   4  W019  29            1.000000001   1.000000001   0.999999997   0.000000003   no_resolved_level4_gap
   5  W017  29            1.000000001   1.000000000   0.999999997   0.000000003   no_resolved_level4_gap
   6  W015  29            1.000000001   1.000000000   0.999999997   0.000000003   no_resolved_level4_gap
   7  W013  29            0.999904431   0.999999996   1.000000000   0.000000000   no_resolved_level4_gap
   8  W014  29            0.999904474   0.999999996   1.000000000   0.000000000   no_resolved_level4_gap
   9  W012  29            0.999904488   0.999999

In [23]:
function installed_version(package_name)
    for package in values(Pkg.dependencies())
        if package.name == package_name
            return package.version === nothing ? "unknown" : string(package.version)
        end
    end
    return "not found"
end

function write_run_metadata(path, candidates, levels, state)
    open(path, "w") do io
        println(io, "Cycle-polytope QuantumNPA screening")
        println(io, "updated_at = ", state["updated_at"])
        println(io, "julia_version = ", VERSION)
        println(io, "julia_threads = ", Threads.nthreads())
        for package in required_packages
            println(io, lowercase(package), "_version = ", installed_version(package))
        end
        println(io, "candidate_count = ", length(candidates))
        println(io, "npa_levels = ", join(levels, ","))
        println(io, "scs_eps_abs = ", SCS_EPS_ABS)
        println(io, "scs_eps_rel = ", SCS_EPS_REL)
        println(io, "scs_max_iters = ", SCS_MAX_ITERS)
        println(io, "screening_tolerance = ", NPA_SCREENING_TOLERANCE)
        println(io, "monotonicity_tolerance = ", NPA_MONOTONICITY_TOLERANCE)
        println(io, "coordinate_order = xyz blocks lexicographic; abc inside each block lexicographic")
        println(io, "cycle_vertex_count = ", size(vertices_cycle, 1))
        println(io, "cycle_affine_dimension = ", affine_dimension(vertices_cycle))
        println(io, "perturbation_input_directory = ", abspath(PERTURBATION_RESULTS_DIRECTORY))
    end
    return path
end

write_run_metadata(
    output_paths.metadata,
    audited_candidates,
    NPA_LEVELS,
    npa_state,
)

println("Metadata:        ", abspath(output_paths.metadata))
println("\nInstalled package versions")
for package in required_packages
    println("  ", rpad(package, 20), installed_version(package))
end


Metadata:        C:\Users\josen\OneDrive\Desktop\Work\Post-doc_Recife\alternative_projects\Rabello's_project\CV_dropbox_content\Projeto_Ze_Gabrielzinho_Rabelo_Planeta\Jupiter\cycle_npa_results_tight_scs\run_metadata.txt

Installed package versions
  JuMP                1.30.1
  MathOptInterface    1.54.0
  GLPK                1.2.1
  QuantumNPA          0.1.2
  SCS                 2.7.0


## What to send back for validation

After the run finishes, send:

1. `cycle_npa_results/quantum_npa_results.tsv`;
2. `cycle_npa_results/run_metadata.txt`;
3. this executed notebook, so that its printed diagnostics are preserved.

The binary `.jls` file is useful for resuming on your computer, but it is not
needed for scientific validation.  The next stage will take the candidates
with a clearly subunit level-4 value and run a see-saw to obtain explicit
quantum realizations.
